<a href="https://colab.research.google.com/github/Ayushmishra05/Seq-2-Seq-Transformer/blob/main/Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
from IPython.display import Image
Image(url="https://towardsdatascience.com/wp-content/uploads/2024/01/1vGeLxPDsEnOsX3WwQeC6hw.png")

In [17]:
import torch.nn as nn
import torch
import torch.nn.functional as F
from einops import rearrange

In [18]:
class MultiHeadAttention(nn.Module):
  def __init__(self, d_model , n_heads, bias = False) -> None:
    super().__init__()
    assert d_model % n_heads == 0 , "d_model not divisible by n_heads"
    self.d_model = d_model
    self.n_heads = n_heads
    self.bias = bias
    self.qw = nn.Linear(self.d_model , self.d_model , bias = self.bias)
    self.kw = nn.Linear(self.d_model , self.d_model , bias = self.bias)
    self.vw = nn.Linear(self.d_model , self.d_model , bias = self.bias)
    self.project = nn.Linear(self.d_model , self.d_model)

  # def cross_forward()

  def forward(self, q , k , v ,  mask = None):
    ## X dimension ==> (batch_size, seq_len, dim)

    ## (batch_dim, seq_len, d_out)
    q = self.qw(q)
    k = self.kw(k)
    v = self.vw(v)

    # (batch seq_len d_out) -> (batch n_heads seq_len d_out)
    q = rearrange(q , "b s (h d) -> b h s d", h = self.n_heads)
    k = rearrange(k , "b s (h d) -> b h s d", h = self.n_heads)
    v = rearrange(v , "b s (h d) -> b h s d", h = self.n_heads)

    attention_scores = (q @ k.transpose(-2 , -1)) / (k.shape[-1]**0.5)

    # if mask:
    #   masks = torch.triu(torch.ones(seq_len , seq_len) , diagonal=1)
    #   attention_scores = attention_scores.masked_fill(masks == 1 , float("-inf"))
    #   attention_weights = F.softmax(attention_scores/(k.shape[-1]**0.5) , dim=-1)
    # else:
    #   attention_weights = F.softmax(attention_scores/(k.shape[-1]**0.5) , dim=-1)


    if mask is not None:
      attention_scores = attention_scores.masked_fill(mask == 0 , -1e9)
    attention_weights = F.softmax(attention_scores , dim = -1)

    context_vector = attention_weights @ v

    context_vector = rearrange(context_vector , "b h s d -> b s (h d)")
    return self.project(context_vector)


In [19]:
## Expansion and Contraction Module

class FeedForward(nn.Module):
  def __init__(self, d_in) -> None:
    super().__init__()
    self.ff = nn.Sequential(
        nn.Linear(d_in , 4 * d_in),
        nn.GELU(),
        nn.Linear(4 * d_in , d_in)
    )

  def forward(self, x):
    return self.ff(x)

In [ ]:
class Encoder(nn.Module):
  def __init__(self, d_model , n_heads , dropout_ratio = 0.2 , bias = False) -> None:
    super().__init__()
    self.attention = MultiHeadAttention(d_model=d_model , n_heads=n_heads , bias = bias)
    self.ff = FeedForward(d_in=d_model)
    self.norm1 = nn.LayerNorm(d_model)
    self.norm2 = nn.LayerNorm(d_model)
    self.dropout = nn.Dropout(dropout_ratio)

  def forward(self , x , src_mask):
    attention_out = self.attention(x , x , x , src_mask)
    x = x + self.norm1(self.dropout(attention_out))
    ff_out = self.ff(x)
    x = x + self.norm2(self.dropout(ff_out))
    return xg

In [21]:
class Decoder(nn.Module):
  def __init__(self, d_model , n_heads , dropout_ratio = 0.2 , bias = False) -> None:
    super().__init__()
    self.attention = MultiHeadAttention(d_model=d_model , n_heads=n_heads , bias = bias)
    self.ff = FeedForward(d_in=d_model)
    self.norm1 = nn.LayerNorm(d_model)
    self.norm2 = nn.LayerNorm(d_model)
    self.norm3 = nn.LayerNorm(d_model)
    self.dropout = nn.Dropout(dropout_ratio)

  def forward(self, x , enc_output , src_mask, tgt_mask):
    attention_outputs = self.attention(x , x, x, tgt_mask)
    x = x + self.norm1(self.dropout(attention_outputs))
    cross_attention_outputs = self.attention(x , enc_output , enc_output ,  src_mask)
    x = x + self.norm2(self.dropout(cross_attention_outputs))
    ff_output = self.ff(x)
    x = x + self.norm3(self.dropout(ff_output))
    return x




In [22]:
Image(url = "https://miro.medium.com/v2/resize:fit:1100/format:webp/0*jCV-3u3T5S4DbUGa.png")

In [23]:
class PositionalEncoding(nn.Module):
  def __init__(self , d_model , max_seq_len):
    super().__init__()
    self.pos = torch.arange(0 , max_seq_len)
    self.theta = 1 / ((10000 ** (torch.arange(0 , d_model , 2))) / d_model)

    pe = torch.zeros(max_seq_len , d_model)

    pe[...,0::2] = torch.sin(self.pos[: , None] / self.theta)
    pe[...,1::2] = torch.cos(self.pos[: , None] / self.theta)

    self.register_buffer("pe" , pe)

  def forward(self, x):
    b , s , d = x.shape
    return x + self.pe[:s , :]

In [24]:
def get_mask(src , tgt):
  src_mask = (src != 0)
  src_mask = src_mask[: , None , None , :]
  tgt_mask = (tgt != 0)[: , None , : , None]
  seq_len = tgt_mask.shape[-2]
  causal_mask = torch.tril(torch.ones(1 , seq_len , seq_len)).bool()
  final_tgt_mask = tgt_mask & causal_mask
  return src_mask , final_tgt_mask


class Transformer(nn.Module):
  def __init__(self , src_vocab_size , d_model , tgt_vocab_size , max_seq_len , n_heads , dropout_ratio , bias , n_encoders , n_decoders) -> None:
    super().__init__()
    self.encod_embedding = nn.Embedding(src_vocab_size, d_model)
    self.decod_embedding = nn.Embedding(tgt_vocab_size, d_model)
    self.encoding = PositionalEncoding(d_model=d_model, max_seq_len=max_seq_len)
    self.encoder = nn.ModuleList([Encoder(d_model=d_model, n_heads=n_heads, dropout_ratio=dropout_ratio , bias = bias) for i in range(n_encoders)])
    self.decoder = nn.ModuleList([Decoder(d_model=d_model, n_heads=n_heads , dropout_ratio=dropout_ratio , bias = bias) for i in range(n_encoders)])
    self.dropout = nn.Dropout(dropout_ratio)
    self.ff = nn.Linear(d_model, tgt_vocab_size)

  def forward(self , src, tgt):
    src_mask , tgt_mask = get_mask(src , tgt)
    encod_embed = self.dropout(self.encoding(self.encod_embedding(src)))
    decod_embed = self.dropout(self.encoding(self.decod_embedding(tgt)))

    enc_output = encod_embed
    for encoder in self.encoder:
      enc_output = encoder(enc_output , src_mask)

    dec_output = decod_embed
    for decoder in self.decoder:
      dec_output = decoder(dec_output , enc_output , src_mask , tgt_mask )

    out = self.ff(dec_output)
    return out


In [68]:
src_vocab_size = 5000
tgt_vocab_size = 5000
max_seq_len = 100
n_heads = 4
d_hidden = 64
dropout_ratio = 0.2
bias = False
n_encoders = 4
n_decoders = 4
d_model = 24

In [77]:
model = Transformer(src_vocab_size , d_model , tgt_vocab_size, max_seq_len , n_heads  , dropout_ratio , bias , n_encoders  , n_decoders)

In [76]:
# "I" : 0 , "Like" : 1 , "Cats" : 2 , "and" : 3 , "Dogs" : 4 , "<PAD>" : 5

In [71]:
# src = torch.randint(0 , 8 , (1 , 4))
# tgt = torch.randint(0 , 8 , (1 , 4))

In [72]:
src , tgt

(tensor([[0, 1, 1, 0]]), tensor([[4, 0, 7, 4]]))

In [74]:
model(src , tgt).shape

torch.Size([1, 4, 8])

In [75]:
torch.argmax(model(src , tgt), dim=-1)

tensor([[4, 0, 2, 1]])

In [50]:
model(src, tgt).shape

torch.Size([1, 6, 5000])